## Bronze — Ingestão com Auto Loader
Lê arquivos novos de um diretório e salva na tabela Delta sem transformações.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ATMOS
  MANAGED LOCATION 'abfss://atmos-landing-dev-001@saatmosdevwestus2001.dfs.core.windows.net/';

CREATE SCHEMA IF NOT EXISTS ATMOS.BRONZE;

CREATE EXTERNAL VOLUME IF NOT EXISTS ATMOS.BRONZE.LANDING
  LOCATION 'abfss://atmos-landing-dev-001@saatmosdevwestus2001.dfs.core.windows.net/landing'

In [0]:
from pyspark.sql import functions as F

### 1. Leitura com Auto Loader
O Auto Loader monitora o diretório e processa apenas arquivos novos.
Arquivos já processados ficam registrados no checkpoint — sem duplicatas em re-runs.

In [0]:
# Criação de widgets para parametrização do pipeline
dbutils.widgets.text("source_system",       "", "Sistema de origem (ex: inmet)")
dbutils.widgets.text("source_path",         "", "Diretório com os arquivos de entrada")
dbutils.widgets.text("file_format",         "", "Formato: csv | json | parquet")
dbutils.widgets.text("target_table",        "", "Tabela de destino (ex: atmos.bronze.inmet_raw)")
dbutils.widgets.text("ingestion_date",      "", "Data de ingestão (YYYY-MM-DD)")
dbutils.widgets.text("schema_location",     "", "Caminho para o Auto Loader salvar o schema")
dbutils.widgets.text("checkpoint_location", "", "Caminho para o checkpoint do Auto Loader")

# Recuperação dos valores informados nos widgets
source_system       = dbutils.widgets.get("source_system")
source_path         = dbutils.widgets.get("source_path")
file_format         = dbutils.widgets.get("file_format")
target_table        = dbutils.widgets.get("target_table")
ingestion_date      = dbutils.widgets.get("ingestion_date")
schema_location     = dbutils.widgets.get("schema_location")
checkpoint_location = dbutils.widgets.get("checkpoint_location")

### 2. Metadados de rastreabilidade
Bronze não transforma dados — só adiciona colunas de controle.

In [0]:
# Leitura de dados em streaming utilizando Auto Loader (cloudFiles)
df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",           file_format)
    .option("cloudFiles.schemaLocation",   schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header",    "true")   # CSV: lê a primeira linha como cabeçalho
    .option("multiLine", "true")   # JSON: lê documentos de múltiplas linhas
    .load(source_path)
)

In [0]:
# Enriquecimento da camada Bronze com metadados de ingestão
df_bronze = (
    df_raw
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_system",       F.lit(source_system))
    .withColumn("file_name",           F.col("_metadata.file_path"))
    .withColumn("ingestion_date",      F.to_date(F.lit(ingestion_date)))
)

### 3. Escrita na tabela Delta
`availableNow=True` processa tudo que está pendente e encerra (modo batch).
`mergeSchema=true` permite que novas colunas sejam incorporadas automaticamente.

In [0]:
# Escrita dos dados na camada Bronze em formato Delta (streaming)
(
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .option("mergeSchema",        "true")
    .partitionBy("ingestion_date")
    .trigger(availableNow=True)
    .toTable(target_table)
    .awaitTermination()
)

### 4. Validação

In [0]:
# Validação da carga: verifica se houve ingestão de registros na tabela de destino
count = (
    spark.table(target_table)
    .filter(F.col("ingestion_date") == ingestion_date)
    .count()
)

assert count > 0, f"Nenhum registro gravado em {target_table} para {ingestion_date}."
print(f"OK — {count} registros em {target_table} | {ingestion_date}")